In [2]:
# Install vLLM. Only torchaudio is removed (CUDA mismatch); torchvision must stay.
!pip install vllm httpx
!pip uninstall -y torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 92

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [1]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
!python3 -c "import torch, torchvision, vllm; print('OK', vllm.__version__, torch.__version__, torchvision.__version__)"

name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB
OK 0.29.0 2.13.0+cu130 0.28.0+cu130


In [13]:
!ls -la bench.py prompts.txt benchmark_payloads.json concurrency_benchmark.py smoke_test_request.json
!wc -l prompts.txt

-rw-r--r-- 1 root root 198485 Sep 19 11:44 benchmark_payloads.json
-rw-r--r-- 1 root root  11531 Sep 19 11:42 bench.py
-rw-r--r-- 1 root root  11318 Sep 19 11:58 concurrency_benchmark.py
-rw-r--r-- 1 root root   1953 Sep 19 11:42 prompts.txt
-rw-r--r-- 1 root root   9323 Sep 19 11:57 smoke_test_request.json
20 prompts.txt


In [14]:
import subprocess, time, httpx, os

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
EAGER = False            # set True only if startup fails
PREFIX_CACHING = True    # set False to measure without the prefix cache

def ready():
    try:
        return httpx.get("http://localhost:8000/v1/models", timeout=2).status_code == 200
    except Exception:
        return False

if ready():
    print("Server is already running. Skip to the next cell.")
else:
    cmd = ["python3", "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL,
           "--quantization", "awq",
           "--dtype", "half",
           "--max-model-len", "16384",
           "--gpu-memory-utilization", "0.85",
           "--port", "8000"]
    if EAGER:
        cmd.append("--enforce-eager")
    if not PREFIX_CACHING:
        cmd.append("--no-enable-prefix-caching")

    env = dict(os.environ, VLLM_USE_FLASHINFER_SAMPLER="0")
    p = subprocess.Popen(cmd, stdout=open("vllm.log", "w"),
                         stderr=subprocess.STDOUT, env=env, start_new_session=True)

    for i in range(60):
        if p.poll() is not None:
            print("Server exited early, exit code:", p.returncode)
            print(subprocess.run("grep 'core.py:1374' vllm.log | tail -6 | cut -c60-400",
                                 shell=True, capture_output=True, text=True).stdout)
            break
        if ready():
            print("Server ready after", i * 10, "seconds")
            break
        time.sleep(10)
    else:
        print("Timed out waiting for the server")

!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-7B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

Server is already running. Skip to the next cell.
[level 1] tok/s=36.07 ttft_p95=0.0951 errors=0
[level 2] tok/s=70.56 ttft_p95=0.0835 errors=0
[level 4] tok/s=132.16 ttft_p95=0.0957 errors=0
[level 8] tok/s=233.49 ttft_p95=0.1373 errors=0
[level 16] tok/s=344.87 ttft_p95=0.2397 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     36.07      0.048      0.095     3.926    20     0
   2     70.56      0.079      0.084     3.626    20     0
   4    132.16      0.086      0.096     3.700    20     0
   8    233.49      0.093      0.137     3.743    20     0
  16    344.87      0.236      0.240     4.492    20     0

wrote bench_report.json (run appended)


In [15]:
!sed -i 's#Qwen/Qwen2.5-7B-Instruct"#Qwen/Qwen2.5-7B-Instruct-AWQ"#' smoke_test_request.json
!curl -s -X POST http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d @smoke_test_request.json | python3 -m json.tool

{
    "id": "chatcmpl-a803f7c227cdf271",
    "object": "chat.completion",
    "created": 1789820838,
    "model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "```json\n{\n\"predicted_tags\": [\"Decision Trees\", \"Gini Impurity\", \"Pruning\"],\n\"difficulty_level\": \"Intermediate\",\n\"confidence\": 0.85,\n\"notes\": \"The content covers the basics of decision trees, including practical implementation, Gini impurity, and pruning techniques. It provides a clear and detailed explanation suitable for intermediate learners.\"\n}\n```",
                "refusal": null,
                "annotations": null,
                "audio": null,
                "function_call": null,
                "reasoning": null
            },
            "logprobs": null,
            "finish_reason": "stop",
            "stop_reason": null,
            "token_ids": null,
            

In [18]:
import json

payloads = json.load(open("benchmark_payloads.json", encoding="utf-8"))
with open("prompts_real.txt", "w", encoding="utf-8") as f:
    for pl in payloads:
        # bench.py reads one prompt per line, so collapse newlines into spaces
        f.write(" ".join(pl["prompt"].split()) + "\n")
print(len(payloads), "real prompts written")

12 real prompts written


In [19]:
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-7B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts_real.txt \
  --max-tokens 400 \
  --out bench_report_real.json

[level 1] tok/s=12.86 ttft_p95=26.7581 errors=0
[level 2] tok/s=50.18 ttft_p95=0.2004 errors=0
[level 4] tok/s=76.14 ttft_p95=0.2426 errors=0
[level 8] tok/s=97.76 ttft_p95=0.4711 errors=0
[level 16] tok/s=115.67 ttft_p95=0.7302 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     12.86      0.945     26.758    31.028    20     0
   2     50.18      0.129      0.200     5.601    20     0
   4     76.14      0.157      0.243     6.447    20     0
   8     97.76      0.264      0.471     8.537    20     0
  16    115.67      0.462      0.730    13.230    20     0

wrote bench_report_real.json (run appended)


In [20]:
from pathlib import Path

p = Path("concurrency_benchmark.py")
src = p.read_text(encoding="utf-8")

if "--files-per-course" in src:
    print("Already patched, nothing to do.")
else:
    old = "    args = ap.parse_args()"
    new = '''    ap.add_argument("--files-per-course", type=int, default=1,
                    help="Average number of content files (requests) per course")
    ap.add_argument("--gpu-hours-per-day", type=float, default=24,
                    help="Hours per day the GPU is billed (24 = always-on instance)")
    args = ap.parse_args()'''
    assert old in src, "parse_args line not found"
    src = src.replace(old, new)

    start = src.index('    print("PART 22')
    end = src.index("    # ---- write CSV ----")
    cost_block = '''    print("PART 22 -- Cost per course (self-hosted GPU)")
    valid = [r for r in table_rows if r["throughput_req_s"]]
    if valid:
        best = max(valid, key=lambda r: r["throughput_req_s"])
        req_per_hour = best["throughput_req_s"] * 3600
        cost_per_request = args.gpu_hourly_cost / req_per_hour
        cost_per_course = cost_per_request * args.files_per_course
        usage_monthly = cost_per_course * args.courses_per_month
        always_on_monthly = args.gpu_hourly_cost * args.gpu_hours_per_day * 30

        print(f"Best throughput at concurrency {best['concurrency']}: "
              f"{best['throughput_req_s']} req/s (~{req_per_hour:.0f} req/hr)")
        print(f"GPU hourly cost:                ${args.gpu_hourly_cost:.2f}/hr")
        print(f"Cost per request:               ${cost_per_request:.5f}")
        print(f"Files per course (assumed):     {args.files_per_course}")
        print(f"Cost per course:                ${cost_per_course:.5f}")
        print(f"Monthly, pay-per-use ({args.courses_per_month} courses): ${usage_monthly:.2f}")
        print(f"Monthly, GPU billed {args.gpu_hours_per_day:g}h/day:     ${always_on_monthly:.2f}")

'''
    src = src[:start] + cost_block + src[end:]
    p.write_text(src, encoding="utf-8")
    print("Patched OK.")

!python3 -m py_compile concurrency_benchmark.py && echo "Syntax OK"

!python3 concurrency_benchmark.py \
  --base-url http://localhost:8000/v1 \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --payloads benchmark_payloads.json \
  --gpu-hourly-cost 1.80 \
  --courses-per-month 200 \
  --files-per-course 12

Already patched, nothing to do.
Syntax OK
Loaded 12 real content payloads (avg ~3914 input tokens each)

GPU before run: 0.0% util, 12123/15360 MB VRAM

--- Concurrency = 1 ---
{
  "n": 20,
  "errors": 0,
  "error_rate_pct": 0.0,
  "avg_latency_s": 4.891,
  "p95_latency_s": 8.884,
  "avg_ttft_s": 1.661,
  "throughput_req_s": 0.204,
  "tokens_per_sec": 21.7,
  "wall_time_s": 97.83,
  "concurrency": 1,
  "gpu_util_pct": 100.0,
  "vram_used_mb": 12123.0
}

--- Concurrency = 2 ---
{
  "n": 20,
  "errors": 0,
  "error_rate_pct": 0.0,
  "avg_latency_s": 4.092,
  "p95_latency_s": 5.716,
  "avg_ttft_s": 0.157,
  "throughput_req_s": 0.467,
  "tokens_per_sec": 49.7,
  "wall_time_s": 42.83,
  "concurrency": 2,
  "gpu_util_pct": 100.0,
  "vram_used_mb": 12123.0
}

--- Concurrency = 5 ---
{
  "n": 20,
  "errors": 0,
  "error_rate_pct": 0.0,
  "avg_latency_s": 6.051,
  "p95_latency_s": 8.373,
  "avg_ttft_s": 0.214,
  "throughput_req_s": 0.784,
  "tokens_per_sec": 82.9,
  "wall_time_s": 25.51,
  "con